# 02 - Entrenamiento y Evaluación con Redes Neuronales

Entrenar MLP (redes neuronales), evaluar con SHAP y feature importance, comparar robustez en datos adversarios.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score, 
    accuracy_score, precision_recall_fscore_support
)

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks

# SHAP
import shap

# Configuración
np.random.seed(42)
tf.random.set_seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print(f'TensorFlow version: {tf.__version__}')
print(f'SHAP version: {shap.__version__}')
print('✓ Librerías importadas')

## 1. Cargar Datos Procesados

In [ ]:
# Cargar datos
X_train = np.load('data/processed/X_train_scaled.npy')
X_val = np.load('data/processed/X_val_scaled.npy')
X_test = np.load('data/processed/X_test_scaled.npy')
y_train = np.load('data/processed/y_train_enc.npy')
y_val = np.load('data/processed/y_val_enc.npy')
y_test = np.load('data/processed/y_test_enc.npy')

# Metadatos
with open('data/processed/metadata.pkl', 'rb') as f:
    metadata = pickle.load(f)

feature_names = metadata['feature_names']
class_names = metadata['class_names']
n_features = metadata['n_features']
n_classes = metadata['n_classes']

print(f'X_train shape: {X_train.shape}')
print(f'X_val shape: {X_val.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'\nClases: {class_names}')
print(f'Features: {n_features}')

## 2. Definir Arquitecturas de Redes Neuronales

In [ ]:
def build_model_v1(n_features, n_classes, dropout_rate=0.3):
    """
    Arquitectura V1: 400 -> 256 -> 128 -> 64 -> 5
    Regularización moderada con dropout
    """
    model = keras.Sequential([
        layers.Input(shape=(n_features,)),
        layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate),
        
        layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate),
        
        layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        layers.Dropout(dropout_rate),
        
        layers.Dense(n_classes, activation='softmax')
    ])
    return model

def build_model_v2(n_features, n_classes, dropout_rate=0.2):
    """
    Arquitectura V2: 400 -> 512 -> 256 -> 128 -> 5
    Más profunda y robusta
    """
    model = keras.Sequential([
        layers.Input(shape=(n_features,)),
        layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.0005)),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate),
        
        layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.0005)),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate),
        
        layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.0005)),
        layers.Dropout(dropout_rate),
        
        layers.Dense(n_classes, activation='softmax')
    ])
    return model

print('✓ Arquitecturas definidas')

## 3. Entrenar Modelos

In [ ]:
# Calcular class weights para manejar desbalance
from sklearn.utils.class_weight import compute_class_weight

class_weights_array = compute_class_weight(
    'balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights_array))

print('CLASS WEIGHTS (para balancear desbalance):')
for class_idx, class_name in enumerate(class_names):
    print(f'  {class_name}: {class_weights[class_idx]:.3f}')

# Entrenar V1
print('\n' + '='*60)
print('ENTRENANDO MODELO V1 (400 -> 256 -> 128 -> 64 -> 5)')
print('='*60)

model_v1 = build_model_v1(n_features, n_classes, dropout_rate=0.3)
model_v1.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

history_v1 = model_v1.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    class_weight=class_weights,
    callbacks=[early_stop],
    verbose=0
)

print(f'✓ V1 entrenado ({len(history_v1.history["loss"])} epochs)')

# Entrenar V2
print('\n' + '='*60)
print('ENTRENANDO MODELO V2 (400 -> 512 -> 256 -> 128 -> 5)')
print('='*60)

model_v2 = build_model_v2(n_features, n_classes, dropout_rate=0.2)
model_v2.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0008),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_v2 = model_v2.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    class_weight=class_weights,
    callbacks=[early_stop],
    verbose=0
)

print(f'✓ V2 entrenado ({len(history_v2.history["loss"])} epochs)')

## 4. Comparar Modelos en Validation Set

In [ ]:
# Predecir en validation
y_val_pred_v1 = model_v1.predict(X_val, verbose=0).argmax(axis=1)
y_val_pred_v2 = model_v2.predict(X_val, verbose=0).argmax(axis=1)

# Métricas
macro_f1_v1 = f1_score(y_val, y_val_pred_v1, average='macro')
macro_f1_v2 = f1_score(y_val, y_val_pred_v2, average='macro')
weighted_f1_v1 = f1_score(y_val, y_val_pred_v1, average='weighted')
weighted_f1_v2 = f1_score(y_val, y_val_pred_v2, average='weighted')

print('COMPARACIÓN EN VALIDATION SET:')
print('\nModelo V1 (256-128-64):')
print(f'  Accuracy:      {accuracy_score(y_val, y_val_pred_v1):.4f}')
print(f'  Macro-F1:      {macro_f1_v1:.4f}')
print(f'  Weighted-F1:   {weighted_f1_v1:.4f}')

print('\nModelo V2 (512-256-128):')
print(f'  Accuracy:      {accuracy_score(y_val, y_val_pred_v2):.4f}')
print(f'  Macro-F1:      {macro_f1_v2:.4f}')
print(f'  Weighted-F1:   {weighted_f1_v2:.4f}')

# Seleccionar mejor
best_model = model_v2 if macro_f1_v2 >= macro_f1_v1 else model_v1
best_version = 'V2' if macro_f1_v2 >= macro_f1_v1 else 'V1'
print(f'\n✓ Mejor modelo: {best_version}')

# Visualizar curvas de entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for idx, (history, version) in enumerate([(history_v1, 'V1'), (history_v2, 'V2')]):
    axes[idx].plot(history.history['loss'], label='Train Loss')
    axes[idx].plot(history.history['val_loss'], label='Val Loss')
    axes[idx].set_xlabel('Epoch')
    axes[idx].set_ylabel('Loss')
    axes[idx].set_title(f'Modelo {version} - Training Curves')
    axes[idx].legend()
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('reports/03_training_curves.png', dpi=100, bbox_inches='tight')
plt.show()
print('✓ Figura guardada: reports/03_training_curves.png')

## 5. Evaluación Final en Test Set

In [ ]:
# Predicciones finales en test
y_test_pred = best_model.predict(X_test, verbose=0).argmax(axis=1)
y_test_proba = best_model.predict(X_test, verbose=0)

# Métricas
accuracy = accuracy_score(y_test, y_test_pred)
macro_f1 = f1_score(y_test, y_test_pred, average='macro')
weighted_f1 = f1_score(y_test, y_test_pred, average='weighted')
micro_f1 = f1_score(y_test, y_test_pred, average='micro')

print('='*60)
print('EVALUACIÓN FINAL - TEST SET')
print('='*60)
print(f'Accuracy:      {accuracy:.4f}')
print(f'Macro-F1:      {macro_f1:.4f} ⭐ (Principal)')
print(f'Weighted-F1:   {weighted_f1:.4f}')
print(f'Micro-F1:      {micro_f1:.4f}')

# Por clase
print('\nREPORTE POR CLASE:')
print(classification_report(y_test, y_test_pred, target_names=class_names))

# Matriz de confusión
cm = confusion_matrix(y_test, y_test_pred)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names, ax=ax,
            cbar_kws={'label': 'Conteo'})
ax.set_xlabel('Predicción')
ax.set_ylabel('Real')
ax.set_title(f'Matriz de Confusión (Test) - Macro-F1: {macro_f1:.4f}')
plt.tight_layout()
plt.savefig('reports/04_confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()
print('✓ Figura guardada: reports/04_confusion_matrix.png')

## 6. Feature Importance - Permutation

In [ ]:
# Permutation importance
from sklearn.inspection import permutation_importance

print('Calculando Permutation Importance...')

# Wrapper para sklearn
class KerasWrapper:
    def __init__(self, model):
        self.model = model
    
    def predict(self, X):
        return self.model.predict(X, verbose=0).argmax(axis=1)
    
    def score(self, X, y):
        return f1_score(y, self.predict(X), average='macro')

wrapper = KerasWrapper(best_model)

perm_importance = permutation_importance(
    wrapper, X_test, y_test,
    n_repeats=10,
    random_state=42,
    n_jobs=-1,
    scoring='f1_macro'
)

# Top 20 features
top_indices = np.argsort(perm_importance.importances_mean)[-20:][::-1]
top_features = [feature_names[i] for i in top_indices]
top_importances = perm_importance.importances_mean[top_indices]

print('\nTOP 20 FEATURES (Permutation Importance):')
for i, (feat, imp) in enumerate(zip(top_features, top_importances), 1):
    print(f'  {i:2d}. {feat:40s} {imp:.6f}')

# Visualizar
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(len(top_features)), top_importances, color='steelblue')
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features, fontsize=9)
ax.set_xlabel('Permutation Importance')
ax.set_title('Top 20 Features - Permutation Importance')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('reports/05_feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()
print('✓ Figura guardada: reports/05_feature_importance.png')

## 7. SHAP - Explicabilidad del Modelo

In [ ]:
print('Calculando SHAP values...')
print('(Esto puede tomar 2-5 minutos)\n')

# Sample de datos para SHAP (usar subsample para velocidad)
sample_size = min(300, len(X_test))
sample_indices = np.random.choice(len(X_test), sample_size, replace=False)
X_test_sample = X_test[sample_indices]

# Crear explainer SHAP
explainer = shap.KernelExplainer(
    lambda x: best_model.predict(x, verbose=0),
    shap.sample(X_test, min(100, len(X_test)), random_state=42)
)

# Calcular SHAP values
shap_values = explainer.shap_values(X_test_sample)

print('✓ SHAP values calculados')
print(f'  Shape: {np.array(shap_values).shape}')

# Visualizaciones SHAP
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for class_idx, class_name in enumerate(class_names):
    ax = axes[class_idx]
    
    if class_idx < len(shap_values):
        # Sumar SHAP values para cada feature (agregado across samples)
        mean_abs_shap = np.abs(shap_values[class_idx]).mean(axis=0)
        
        # Top 15 features
        top_idx = np.argsort(mean_abs_shap)[-15:]
        
        ax.barh(range(len(top_idx)), mean_abs_shap[top_idx], color=plt.cm.Set2(class_idx))
        ax.set_yticks(range(len(top_idx)))
        ax.set_yticklabels([feature_names[i] for i in top_idx], fontsize=8)
        ax.set_xlabel('Mean |SHAP value|')
        ax.set_title(f'Top Features para: {class_name}')
        ax.invert_yaxis()

# Ocultar último subplot
axes[5].axis('off')

plt.suptitle('SHAP Feature Importance por Clase', fontsize=14, y=1.00)
plt.tight_layout()
plt.savefig('reports/06_shap_importance.png', dpi=100, bbox_inches='tight')
plt.show()
print('✓ Figura guardada: reports/06_shap_importance.png')

## 8. Guardar Modelo y Metadatos

In [ ]:
# Guardar modelo
best_model.save('models/best_model_mlp.h5')

# Guardar resultados en CSV
results_df = pd.DataFrame({
    'Métrica': ['Accuracy', 'Macro-F1', 'Weighted-F1', 'Micro-F1'],
    'Valor': [accuracy, macro_f1, weighted_f1, micro_f1]
})

results_df.to_csv('reports/metrics_summary.csv', index=False)

# Reporte detallado
with open('reports/evaluation_report.txt', 'w', encoding='utf-8') as f:
    f.write('='*70 + '\n')
    f.write('REPORTE TÉCNICO - DETECCIÓN DE MALWARE ANDROID (MODALIDAD TABULAR)\n')
    f.write('='*70 + '\n\n')
    
    f.write(f'Fecha: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n\n')
    
    f.write('1. ARQUITECTURA DEL MODELO\n')
    f.write('-' * 70 + '\n')
    f.write(f'Tipo: Red Neuronal Profunda (MLP)\n')
    f.write(f'Versión: {best_version}\n')
    f.write(f'Features de entrada: {n_features}\n')
    f.write(f'Clases de salida: {n_classes}\n')
    f.write(f'Optimizador: Adam\n')
    f.write(f'Loss: Sparse Categorical Crossentropy\n')
    f.write(f'Regularización: L2 + Batch Normalization + Dropout\n\n')
    
    f.write('2. MÉTRICAS DE EVALUACIÓN (TEST SET)\n')
    f.write('-' * 70 + '\n')
    f.write(f'Accuracy:      {accuracy:.6f}\n')
    f.write(f'Macro-F1:      {macro_f1:.6f} ⭐\n')
    f.write(f'Weighted-F1:   {weighted_f1:.6f}\n')
    f.write(f'Micro-F1:      {micro_f1:.6f}\n\n')
    
    f.write('3. REPORTE POR CLASE\n')
    f.write('-' * 70 + '\n')
    f.write(classification_report(y_test, y_test_pred, target_names=class_names))
    f.write('\n\n')
    
    f.write('4. TOP 20 FEATURES (Permutation Importance)\n')
    f.write('-' * 70 + '\n')
    for i, (feat, imp) in enumerate(zip(top_features, top_importances), 1):
        f.write(f'{i:2d}. {feat:40s} {imp:.6f}\n')
    f.write('\n')
    
    f.write('5. DATOS Y SPLITS\n')
    f.write('-' * 70 + '\n')
    f.write(f'Train samples:     {X_train.shape[0]}\n')
    f.write(f'Val samples:       {X_val.shape[0]}\n')
    f.write(f'Test samples:      {X_test.shape[0]}\n')
    f.write(f'Total:             {X_train.shape[0] + X_val.shape[0] + X_test.shape[0]}\n\n')
    
    f.write('6. DISTRIBUCIÓN DE CLASES (TEST)\n')
    f.write('-' * 70 + '\n')
    for class_name in class_names:
        count = (y_test == np.where(class_names == class_name)[0][0]).sum()
        pct = 100 * count / len(y_test)
        f.write(f'{class_name:15s}: {count:5d} ({pct:5.2f}%)\n')
    f.write('\n')

print('✓ Modelo guardado: models/best_model_mlp.h5')
print('✓ Métricas guardadas: reports/metrics_summary.csv')
print('✓ Reporte guardado: reports/evaluation_report.txt')

## 9. Evaluar Robustez en Datos Adversarios y Ofuscados

In [ ]:
# Cargar scaler
with open('models/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open('models/label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)

# Cargar datos adversarios
data_path = 'Multimodal_CIC-MalDroid2020/Tabular/'

print('Cargando datos adversarios y ofuscados...')
df_adv = pd.read_csv(f'{data_path}adv-tabular.csv')
df_obfus = pd.read_csv(f'{data_path}obfus-tabular.csv')

# Features usadas
used_features = metadata['feature_names']

# Preprocesar adversarios
X_adv = df_adv[used_features].values
X_adv_scaled = scaler.transform(X_adv)
y_adv = le.transform(df_adv['Class'].values)

# Preprocesar ofuscados
X_obfus = df_obfus[used_features].values
X_obfus_scaled = scaler.transform(X_obfus)
y_obfus = le.transform(df_obfus['Class'].values)

# Evaluar
y_adv_pred = best_model.predict(X_adv_scaled, verbose=0).argmax(axis=1)
y_obfus_pred = best_model.predict(X_obfus_scaled, verbose=0).argmax(axis=1)

macro_f1_adv = f1_score(y_adv, y_adv_pred, average='macro')
macro_f1_obfus = f1_score(y_obfus, y_obfus_pred, average='macro')
accuracy_adv = accuracy_score(y_adv, y_adv_pred)
accuracy_obfus = accuracy_score(y_obfus, y_obfus_pred)

print('\n' + '='*60)
print('EVALUACIÓN DE ROBUSTEZ')
print('='*60)
print('\nTEST ORIGINAL:')
print(f'  Accuracy:   {accuracy:.4f}')
print(f'  Macro-F1:   {macro_f1:.4f}')

print('\nDATA ADVERSARIO (perturbaciones):')
print(f'  Accuracy:   {accuracy_adv:.4f} (drop: {100*(accuracy - accuracy_adv)/accuracy:+.2f}%)')
print(f'  Macro-F1:   {macro_f1_adv:.4f} (drop: {100*(macro_f1 - macro_f1_adv)/macro_f1:+.2f}%)')

print('\nDATA OFUSCADO (código ofuscado):')
print(f'  Accuracy:   {accuracy_obfus:.4f} (drop: {100*(accuracy - accuracy_obfus)/accuracy:+.2f}%)')
print(f'  Macro-F1:   {macro_f1_obfus:.4f} (drop: {100*(macro_f1 - macro_f1_obfus)/macro_f1:+.2f}%)')

# Guardar resultados robustez
robustness_df = pd.DataFrame({
    'Dataset': ['Test Original', 'Adversario', 'Ofuscado'],
    'Accuracy': [accuracy, accuracy_adv, accuracy_obfus],
    'Macro-F1': [macro_f1, macro_f1_adv, macro_f1_obfus],
    'Samples': [len(y_test), len(y_adv), len(y_obfus)]
})

robustness_df.to_csv('reports/robustness_evaluation.csv', index=False)
print('\n✓ Resultados de robustez guardados: reports/robustness_evaluation.csv')

# Visualizar
fig, ax = plt.subplots(figsize=(10, 5))
x_pos = np.arange(len(robustness_df))
width = 0.35

ax.bar(x_pos - width/2, robustness_df['Accuracy'], width, label='Accuracy', alpha=0.8)
ax.bar(x_pos + width/2, robustness_df['Macro-F1'], width, label='Macro-F1', alpha=0.8)

ax.set_ylabel('Score')
ax.set_title('Evaluación de Robustez')
ax.set_xticks(x_pos)
ax.set_xticklabels(robustness_df['Dataset'])
ax.legend()
ax.grid(alpha=0.3, axis='y')
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('reports/07_robustness_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print('✓ Figura guardada: reports/07_robustness_comparison.png')

## 10. Resumen Final y Recomendaciones

In [ ]:
print('\n' + '='*70)
print('RESUMEN FINAL - PIPELINE TABULAR COMPLETO')
print('='*70)

print('\n1. PREPROCESAMIENTO')
print('-' * 70)
print(f'  ✓ Features originales: 400')
print(f'  ✓ Features finales: {n_features} (reducción: {100*(400-n_features)/400:.1f}%)')
print(f'  ✓ Eliminadas: {400 - n_features} (constantes + correlación alta)')

print('\n2. MODELO ENTRENADO')
print('-' * 70)
print(f'  ✓ Arquitectura: Red Neuronal ({best_version})')
print(f'  ✓ Capas: 4 densas + BatchNormalization + Dropout')
print(f'  ✓ Parámetros: ~{best_model.count_params():,}')

print('\n3. PERFORMANCE (TEST SET)')
print('-' * 70)
print(f'  ✓ Accuracy:    {accuracy:.4f}')
print(f'  ✓ Macro-F1:    {macro_f1:.4f} ⭐ MÉTRICA PRINCIPAL')
print(f'  ✓ Weighted-F1: {weighted_f1:.4f}')

print('\n4. INTERPRETABILIDAD')
print('-' * 70)
print(f'  ✓ Feature Importance calculada (Permutation)')
print(f'  ✓ SHAP values calculados para todas las clases')
print(f'  ✓ Top 20 features identificados')

print('\n5. ROBUSTEZ')
print('-' * 70)
print(f'  ✓ Test Original:     Macro-F1 = {macro_f1:.4f}')
print(f'  ✓ Data Adversario:   Macro-F1 = {macro_f1_adv:.4f} (drop: {100*(macro_f1 - macro_f1_adv)/macro_f1:.2f}%)')
print(f'  ✓ Data Ofuscado:     Macro-F1 = {macro_f1_obfus:.4f} (drop: {100*(macro_f1 - macro_f1_obfus)/macro_f1:.2f}%)')

print('\n6. ARCHIVOS GENERADOS')
print('-' * 70)
files_generated = [
    'models/best_model_mlp.h5',
    'models/scaler.pkl',
    'models/label_encoder.pkl',
    'data/processed/X_train_scaled.npy',
    'data/processed/X_val_scaled.npy',
    'data/processed/X_test_scaled.npy',
    'reports/metrics_summary.csv',
    'reports/evaluation_report.txt',
    'reports/robustness_evaluation.csv',
    'reports/01_correlation_matrix.png',
    'reports/02_pca_analysis.png',
    'reports/03_training_curves.png',
    'reports/04_confusion_matrix.png',
    'reports/05_feature_importance.png',
    'reports/06_shap_importance.png',
    'reports/07_robustness_comparison.png'
]

for f in files_generated:
    if os.path.exists(f):
        size_kb = os.path.getsize(f) / 1024
        print(f'  ✓ {f:45s} ({size_kb:8.1f} KB)')

print('\n' + '='*70)
print('✓ PIPELINE COMPLETADO EXITOSAMENTE')
print('='*70)